# Previously during bundle annotation, we categorised bundles.
# Now, we tokenise items.

### For the Electronics domain: Sometimes items have extremely long product titles like Wasabi Power Battery for Canon LP-E6 and Canon EOS 5D Mark II, EOS 5D Mark III, EOS 6D, EOS 7D, EOS 60D, EOS 60Da, EOS 70D.

### Thus, for an LLM to retrieve them, it's easier for an LLM to describe the characteristics of this item, rather than provide a similar word string.

In [10]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

MessageError: Error: credential propagation was unsuccessful

In [1]:
import pandas as pd
import numpy as np
import csv
import pickle
import copy
import re
import random
import matplotlib.pyplot as plt
import itertools
import json
import openai
import time


In [44]:
!rm -rf LLM4BEAR
!git clone --depth 1 --filter=blob:none --sparse https://github.com/anon5159753/LLM4BEAR.git
!cd LLM4BEAR && git sparse-checkout set "BundleRec Data"

Cloning into 'LLM4BEAR'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 20 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 7.52 KiB | 7.52 MiB/s, done.
remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (1/1), 66 bytes | 66.00 KiB/s, done.
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 73 (delta 16), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 45.00 MiB | 9.98 MiB/s, done.
Resolving deltas: 100% (16/16), done.
Updating files: 100% (74/74), done.


In [ ]:
from google.colab import userdata
my_secret_key = userdata.get('API_KEY')

# wandb_secret_key = userdata.get('WANDB_KEY')

if my_secret_key:
  print("Token retrieved successfully.")
else:
  print("Token not found in Colab Secrets.")

from openai import OpenAI

client = OpenAI(
    # This is the default and can be omitted
    api_key = my_secret_key,
)


import asyncio
from openai import AsyncOpenAI

async_client = AsyncOpenAI(api_key = my_secret_key,
)  # make sure this is your actual key

In [3]:
clothing_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/clothing/bundle_list_items.pkl")

electronic_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/electronic/bundle_list_items.pkl")

food_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/food/bundle_list_items.pkl")

bundle_items_list = [clothing_bundles_items, electronic_bundles_items, food_bundles_items]

clothing_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/bundle_intent.csv")

electronics_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/bundle_intent.csv")

food_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/bundle_intent.csv")

intents = [clothing_intent, electronics_intent, food_intent]

clothing_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/merged_metadata.csv")

electronic_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/merged_metadata.csv")

food_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/merged_metadata.csv")

metadata = [clothing_metadata, electronic_metadata, food_metadata]

clothing_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_session.csv")

clothing_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_item.csv")

clothing_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_bundle.csv")

clothing_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/session_bundle.csv")

clothing_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/session_item.csv")

clothing_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/item_titles.csv")

electronic_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_session.csv")

electronic_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_item.csv")

electronic_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_bundle.csv")

electronic_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/session_bundle.csv")

electronic_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/session_item.csv")

electronic_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/item_titles.csv")

food_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_session.csv")

food_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_item.csv")

food_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_bundle.csv")

food_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/session_bundle.csv")

food_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/session_item.csv")

food_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/item_titles.csv")

user_session = [clothing_user_session, electronic_user_session, food_user_session]

user_item = [clothing_user_item, electronic_user_item, food_user_item]

user_bundle = [clothing_user_bundle, electronic_user_bundle, food_user_bundle]

session_bundle = [clothing_session_bundle, electronic_session_bundle, food_session_bundle]

session_item = [clothing_session_item, electronic_session_item, food_session_item]

item_names = [clothing_item_names, electronic_item_names, food_item_names]

In [4]:
with open("/content/LLM4BEAR/BundleRec Data/enriched_outputs_electronic.pkl", "rb") as f:
    text_electronics = pickle.load(f)

print(text_electronics[3498])

with open(f"/content/LLM4BEAR/BundleRec Data/enriched_outputs_clothing.pkl", "rb") as f:
    text_clothing = pickle.load(f)

print(text_clothing[3000])

with open(f"/content/LLM4BEAR/BundleRec Data/enriched_outputs_food.pkl", "rb") as f:
    text_food = pickle.load(f)

print(text_food[3000])


This device is a plug-and-play USB adapter that provides 5.1 channel surround sound capabilities to computers without the need for an internal sound card.
This is a pink swimsuit designed for girls aged 7 to 16, featuring a sweetheart neckline and suitable for swimming and beach activities.
A fragrant blend of star anise, cloves, Chinese cinnamon, Sichuan peppercorns, and ginger, this seasoning enhances a variety of dishes with its unique sweet and savory flavor profile.


In [5]:
def bundle_origin(bundle_ID, domain):

    k = 0
    if domain == "clothing":
        k += 0
    elif domain == "electronic":
        k += 1
    elif domain == "food":
        k += 2

    original_session_ID = session_bundle[k].iloc[bundle_ID]["session ID"]

    item_ids = session_item[k][session_item[k]["session ID"] == original_session_ID]["item ID"].values


    # descript_sess = [metadata[k][metadata[k]['item ID'] == item_id].index.tolist()[0] for item_id in item_ids]

    descript_sess = [
    metadata[k][metadata[k]['item ID'] == item_id].index[0]
    for item_id in item_ids
    if (metadata[k]['item ID'] == item_id).any()
]


    return descript_sess


def bundle_str_generator(bundle_ID, domain, desc):


    k = 0
    if domain == "clothing":
        k += 0
    elif domain == "electronic":
        k += 1
    elif domain == "food":
        k += 2

    bundle_list = bundle_items_list[k]

    descript_ids = [metadata[k][metadata[k]['titles'] == names].index.tolist()[0] for names in bundle_list[bundle_ID]]

    bundle_items =  [metadata[k].iloc[i]['titles'] for i in descript_ids]

    bundle_descriptions = [desc[i] for i in descript_ids]

    bundle_categories = [metadata[k].iloc[i]['categories'] for i in descript_ids]


    bundle_str = ""

    for i in range(len(bundle_items)):
        bundle_str = bundle_str + f"{i+1}. " + bundle_items[i] + ": " + bundle_descriptions[i] + "\n"

    return bundle_str, bundle_categories

In [ ]:
async def single_request(user, system=None, seed_value=None):

    if system:
        message = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    else:
        message = [{"role": "user", "content": user}]

    # Reimplemented the robust retry loop with exponential backoff
    for delay_secs in (2**x for x in range(0, 3)):
        try:
            response = await async_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=message,
                temperature=0,
                max_tokens=8000,
                seed=seed_value
            )
            return response.choices[0].message.content.strip()
        except openai.OpenAIError as e:
            randomness_collision_avoidance = random.randint(0, 1000) / 1000.0
            sleep_dur = delay_secs + randomness_collision_avoidance
            print(f"Error: {e}. Retrying in {round(sleep_dur, 2)} seconds.")
            await asyncio.sleep(sleep_dur)

    # Return None if all retries fail
    return None


async def openai_request(prompts, system=None, batch_size=128, delay=0):

    results = []

    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        tasks = [
            single_request(d["prompts"], system=system, seed_value=42)
            for j, d in enumerate(batch)
        ]

        batch_results = await asyncio.gather(*tasks)
        results.extend(batch_results)
        print(f"✅ Sending batch {i // batch_size + 1} — sleeping for {delay}s...\n")
        await asyncio.sleep(delay)

    return results


In [6]:
def extract_json_simple_replace(response_text):
    """
    Extracts a JSON object from a string that has a "===JSON_START===" separator.

    This function isolates the JSON by finding the first '{' and last '}'
    to ensure it works correctly even with markdown fences or extra whitespace.

    Args:
        response_text (str): The full string containing the separator and JSON.

    Returns:
        dict: The parsed JSON object as a Python dictionary, or None if an error occurs.
    """
    try:
        # 1. Get the text after the separator
        json_part = response_text.split("===JSON_START===")[1]

        # 2. Find the boundaries of the JSON object
        first_brace = json_part.find('{')
        last_brace = json_part.rfind('}')

        # 3. Slice the string to get only the valid JSON
        # This will fail gracefully in the json.loads() if a brace isn't found
        json_string = json_part[first_brace : last_brace + 1]

        # 4. Parse the clean string
        parsed_json = json.loads(json_string)
        return parsed_json

    except IndexError:
        print("Error: The separator '===JSON_START===' was not found.")
        return None
    except json.JSONDecodeError:
        print("Error: Could not find or parse a valid JSON object after the separator.")
        return None


def get_json_list(dump):
    data = []
    for i in range(len(dump)):
        extracted_data = extract_json_simple_replace(dump[i] )
        data.append(extracted_data)
    return data

In [7]:
def input_strings(intent_list, item_list, description_list):
    l = len(intent_list)
    string_list = []
    for i in range(l):
        intent = intent_list[i]
        items = item_list[i]
        descriptions = description_list[i]
        item_str = "\n".join([f"{i + 1}. {title}" for i, title in enumerate(items)])

        string_list.append(f"Intent: {intent}\nBundle Items:\n{item_str}\n")
    return string_list

In [8]:
with open(f"/content/LLM4BEAR/BundleRec Data/intent_categories_electronic.pkl", 'rb') as f:
    final_electronic_intent_guys = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/intent_categories_clothing.pkl", 'rb') as f:
    final_clothing_intent_guys = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/intent_categories_food.pkl", 'rb') as f:
    final_food_intent_guys = pickle.load(f)

print(final_electronic_intent_guys[0])
print()
print(final_clothing_intent_guys[0])
print()
print(final_food_intent_guys[0])

To distill the intents into broader categories, I will analyze the frequency of similar intents and group them accordingly. The goal is to create a concise list of categories that encapsulate the various intents while maintaining a focus on the most frequently mentioned items.

1. **Camera and Accessories**: This category appears most frequently, indicating a strong interest in photography-related products, including cameras, lenses, and various accessories.
  
2. **Computers and Accessories**: This category includes laptops, desktops, and their components and peripherals. It is also mentioned multiple times, reflecting a significant focus on computing devices.

3. **Audio Equipment**: This category encompasses audio devices, sound equipment, and related accessories. It appears frequently, indicating a strong interest in audio technology.

4. **Tablets and Accessories**: Tablets and their accessories are mentioned often, showing a clear demand for portable computing devices.

5. **Stor

In [9]:
final_electronic_intent_json = get_json_list(final_electronic_intent_guys)
true_electronic_intents = final_electronic_intent_json[0]['final_categories'].replace("- [", "").replace("]", "").replace("- ", "").split("\n")

final_clothing_intent_json = get_json_list(final_clothing_intent_guys)
true_clothing_intents = final_clothing_intent_json[0]['final_categories'].replace("- [", "").replace("]", "").replace("- ", "").split("\n")
true_clothing_intents.append("Children's Items")
true_clothing_intents.append("Carrying Items")
true_clothing_intents.append("Maintenance")


final_food_intent_json = get_json_list(final_food_intent_guys)
true_food_intents = final_food_intent_json[0]['final_categories'].replace("- [", "").replace("]", "").replace("- ", "").split("\n")

for i in true_electronic_intents:

    print(i)

print()

for i in true_clothing_intents:

    print(i)

print()

for i in true_food_intents:

    print(i)

Camera and Accessories
Computers and Accessories
Audio Equipment
Tablets and Accessories
Storage Solutions
Networking Equipment
Mobile Devices and Accessories
Travel Accessories
Gaming
Home Entertainment Systems
Miscellaneous Electronics
Power Solutions
Cables and Connectors
Security Systems
Car Technology and Accessories
Photography and Camera Equipment
Adapters and Cables
Television and Accessories
AV Setup
GPS and Navigation Accessories
Mobile Device Protection
Streaming and Media
PC Building and Assembly
General Electronics
Walkie Talkies and Communication Devices

Clothing
Footwear
Accessories
Costumes and Themed Apparel
Lingerie and Underwear
Baby and Kids Clothing
Activewear and Sportswear
Fashion Accessories
Seasonal and Thematic Products
Electronics
Children's Items
Carrying Items
Maintenance

Snacks
Beverages
Cooking Ingredients
Breakfast Foods
Sweets and Desserts
Health Foods
Canned and Packaged Foods
Baby Food
Condiments and Sauces
Fruits and Vegetables
Specialty Foods
Drie

In [ ]:
def finding_food_categories_prompts(items, description_list, final_intent_list):

    prompt_list = []

    categories_str = "\n".join([f"{i + 1}. {title}" for i, title in enumerate(final_intent_list)])
    for i in range(len(items)):
        user_prompt = f"""
**CONTEXT:**
You have received the following item from the food domain. Your goal is to figure out the main category which the item belongs to. An item can belong to more than one category. However, **the more specific it is the better. If an item fits a specific category (e.g., "Sweets and Desserts"), do not also classify it as a general category like "Miscellaneous" unless it genuinely fits both descriptions.**
**Categories:**
{categories_str}

---
**YOUR TASK:**

Categorise the item:
{items[i]}
Description: {description_list[i]}

---

After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions. Do not include any other text after the separator.

**JSON Schema:**
```json
{{
  "Snacks": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Beverages": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Cooking Ingredients": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Breakfast Foods": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Sweets and Desserts": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Health Foods": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Canned and Packaged Foods": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Baby Food": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Condiments and Sauces": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Fruits and Vegetables": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Specialty Foods": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Dried and Preserved Foods": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Grains and Pasta": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Miscellaneous": "integer, 1 if the bundle belongs in no other category, 0 otherwise",
  "Gift Baskets and Food Gifts": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Dietary Specific Items": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Cooking Tools and Kitchen Goods": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Sweeteners": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Nuts and Seeds": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Coffee/Tea": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Vegetables and Beans": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Health-Conscious Options": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Prepared and Ready-Made Meals": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Ethnic and Specialty Foods": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Culinary Specialties": "integer, 1 if the bundle belongs in this category, 0 otherwise"
}}"""
        prompt_list.append(user_prompt)

    return prompt_list

In [ ]:
def finding_clothing_categories_prompts(items, description_list, final_intent_list):

    prompt_list = []

    categories_str = "\n".join([f"{i + 1}. {title}" for i, title in enumerate(final_intent_list)])
    for i in range(len(items)):
        user_prompt = f"""
**CONTEXT:**
You have received the following item from the clothing domain. Your goal is to figure out the main category which the item belongs to. An item can belong to more than one category. However, **the more specific it is the better. If an item fits a specific category (e.g., "Activewear"), do not also classify it as the general "Clothing" unless it genuinely fits both descriptions.**
**Categories:**
{categories_str}

---
**YOUR TASK:**

Categorise the item:
{items[i]}
Description: {description_list[i]}

---

After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions. Do not include any other text after the separator.

**JSON Schema:**
```json
{{
  "Footwear": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Accessories": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Costumes and Themed Apparel": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Lingerie and Underwear": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Baby and Kids Clothing": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Activewear and Sportswear": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Fashion Accessories": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Seasonal and Thematic Products": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Electronics": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Children's Items": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Carrying Items": "integer, 1 if the belongs in this category, 0 otherwise",
  "Maintenance": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Clothing": "integer, 1 if the bundle belongs in no other category, 0 otherwise"
}}
"""
        prompt_list.append(user_prompt)

    return prompt_list

In [ ]:
def finding_electronic_categories_prompts(items, description_list, final_intent_list):

    prompt_list = []

    categories_str = "\n".join([f"{i + 1}. {title}" for i, title in enumerate(final_intent_list)])
    for i in range(len(items)):
        user_prompt = f"""
**CONTEXT:**
You have received the following item from the electronics domain. Your goal is to figure out the main category which the item belongs to. An item can belong to more than one category. However, the more specific it is the better.
**Categories:**
{categories_str}

---
**YOUR TASK:**

Categorise the item:
{items[i]}
Description: {description_list[i]}

---

After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions. Do not include any other text after the separator.

**JSON Schema:**
```json
{{
  "Camera and Accessories": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Computers and Accessories": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Audio Equipment": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Tablets and Accessories": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Storage Solutions": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Networking Equipment": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Mobile Devices and Accessories": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Travel Accessories": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Gaming": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Home Entertainment Systems": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Miscellaneous Electronics": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Power Solutions": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Cables and Connectors": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Security Systems": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Car Technology and Accessories": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Photography and Camera Equipment": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Adapters and Cables": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Television and Accessories": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "AV Setup": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "GPS and Navigation Accessories": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Mobile Device Protection": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Streaming and Media": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "PC Building and Assembly": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "General Electronics": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Walkie Talkies and Communication Devices": "integer, 1 if the bundle belongs in this category, 0 otherwise"
}}
"""

        prompt_list.append(user_prompt)

    return prompt_list

In [11]:


electronic_items = [electronic_metadata.iloc[i]['titles'] for i in range(len(text_electronics))]
clothing_items = [clothing_metadata.iloc[i]['titles'] for i in range(len(text_clothing))]
food_items = [food_metadata.iloc[i]['titles'] for i in range(len(text_food))]

In [12]:
for i in range(20):
  print(food_items[i], ":\n", text_food[i], "\n\n")

Plocky's Hummus Chips, Original, 1-Ounce (Pack of 24) :
 A pack of 24 individual 1-ounce bags of crunchy chips made from hummus, offering a healthier alternative to traditional snack chips. 


Stash Tea Fusion Red, White &amp; Blueberry Tea, 18 Count Tea Bags in Foil (Pack of 6) :
 A pack of 6 boxes containing 18 foil-wrapped tea bags of a fruity red, white, and blueberry blend, ideal for a refreshing beverage. 


Rold Gold Classic Stick Pretzels, 16 Ounce Bag :
 A 16-ounce bag of crunchy, classic stick pretzels perfect for snacking or pairing with dips. 


Twizzlers Pull 'n' Peel Candy, Cherry, 14-Ounce Bags (Pack of 6) :
 This product consists of six 14-ounce bags of cherry-flavored, pull-apart gummy candy that can be enjoyed as a fun and chewy treat. 


Stretch Island Fruit Leather Variety Pack 48-Count, 0.5-Ounce Package :
 This variety pack contains 48 individual 0.5-ounce pieces of fruit leather made from real fruit, offering a convenient and chewy snack option. 


Trident Splash

In [13]:
for i in range(20):
  print(clothing_items[i], ":\n", text_clothing[i], "\n\n")

Tommy Hilfiger Men's Tommy Star Print Boxer :
 Men's boxers featuring a star print design from a well-known brand, offering comfort and style for everyday wear. 


Moving Comfort Women's Luna Bra :
 This bra is designed for women, offering support and comfort with a focus on active lifestyles, featuring moisture-wicking fabric and a stylish, supportive design. 


Patty Women's Cotton Asymmetrical Neck Long Sleeve Stretch Solid Blouse :
 This blouse features a unique asymmetrical neckline, long sleeves, and is made of stretchable cotton fabric, offering both comfort and style for casual or semi-formal occasions. 


GT-Dress Women Fashion Loose plus size strap twinset t shirt blouse top, Women cotton T-shirts Suits :
 This is a fashionable, loose-fitting twinset for women, featuring a strap design and made from cotton, suitable for casual wear as a top and T-shirt combination in plus sizes. 


MA by Michael Antonio Women's Tipton-GLT Platform Sandal :
 This is a stylish platform sandal d

In [14]:
for i in range(20):
  print(electronic_items[i], ":\n", text_electronics[i], "\n\n")

ASUS GTX760-DC2OC-2GD5 GeForce GTX760 2GB GDDR5 256-bit, DVI-I/DVI-D/ HDMI/DP PCI-Express 3.0 SLI ready Graphic Card OC-selected 1072 MHz core :
 A high-performance graphics card designed for gaming and multimedia tasks, featuring 2GB GDDR5 memory, multiple output options, and overclocked speeds for enhanced visual experiences. 


EN-EL15 Battery Charger for Nikon 1 V1, D60,0 D610, D800, D810, D7000, D7100 Digital SLR Camera + More!! :
 A compatible battery charger designed for Nikon digital SLR cameras, including models like the 1 V1, D60, D610, D800, D810, D7000, and D7100. 


Plugable USB 2.0 Flash Memory Card Reader for Windows, Mac, Linux, and Certain Android Systems - Supports SD cards (SDHC, Mini SD, Micro SD / T-Flash, etc) and MS, MS Pro Duo, MMC, More :
 A versatile card reader that enables data transfer from various types of memory cards, including SD, MS, and MMC, to devices running Windows, Mac, Linux, and compatible Android systems. 


Purosol All Natural Lens Cleaner 1oz

In [ ]:
finding_item_electronic_category_prompts = finding_electronic_categories_prompts(electronic_items, text_electronics, true_electronic_intents)

finding_item_electronic_category_prompts = [{"prompts": data} for data in finding_item_electronic_category_prompts]


finding_item_clothing_category_prompts = finding_clothing_categories_prompts(clothing_items, text_clothing, true_clothing_intents)

finding_item_clothing_category_prompts = [{"prompts": data} for data in finding_item_clothing_category_prompts]



finding_item_food_category_prompts = finding_food_categories_prompts(food_items, text_food, true_food_intents)

finding_item_food_category_prompts = [{"prompts": data} for data in finding_item_food_category_prompts]

In [ ]:
# # categories_for_each_electronic_item = await openai_request(finding_item_category_prompts, system=None, batch_size=128, delay=0)

# with open(f"/content/drive/MyDrive/BundleRec Data/item_categories_electronic.pkl", 'wb') as f:
#     pickle.dump(categories_for_each_electronic_item, f)

# categories_for_each_clothing_item = await openai_request(finding_item_clothing_category_prompts, system=None, batch_size=128, delay=0)

# with open(f"/content/drive/MyDrive/BundleRec Data/item_categories_clothing.pkl", 'wb') as f:
#     pickle.dump(categories_for_each_clothing_item, f)

# categories_for_each_food_item = await openai_request(finding_item_food_category_prompts, system=None, batch_size=128, delay=0)

# with open(f"/content/drive/MyDrive/BundleRec Data/item_categories_food.pkl", 'wb') as f:
#     pickle.dump(categories_for_each_food_item, f)

In [16]:
with open(f"/content/LLM4BEAR/BundleRec Data/item_categories_electronic.pkl", 'rb') as f:
    electronic_item_categories = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/item_categories_clothing.pkl", 'rb') as f:
    clothing_item_categories = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/item_categories_food.pkl", 'rb') as f:
    food_item_categories = pickle.load(f)

In [19]:
def extract_json_from_response(response_text):
    # Find the index of the first '{' and the last '}'
    start_index = response_text.find('{')
    end_index = response_text.rfind('}')

    if start_index != -1 and end_index != -1:
        # Slice the string to get only the content between the brackets
        json_string = response_text[start_index : end_index + 1]
        return json_string
    else:
        # Return None if a valid JSON block is not found
        return None

In [21]:
electronic_json_categories = [extract_json_from_response(i) for i in electronic_item_categories]

clothing_json_categories = [extract_json_from_response(i) for i in clothing_item_categories]

food_json_categories = [extract_json_from_response(i) for i in food_item_categories]

print(food_json_categories[3255])

food_json_categories[3255] = """{
  "Snacks": 1,
  "Beverages": 0,
  "Cooking Ingredients": 0,
  "Breakfast Foods": 0,
  "Sweets and Desserts": 0,
  "Health Foods": 0,
  "Canned and Packaged Foods": 0,
  "Baby Food": 0,
  "Condiments and Sauces": 0,
  "Fruits and Vegetables": 0,
  "Specialty Foods": 0,
  "Dried and Preserved Foods": 0,
  "Grains and Pasta": 0,
  "Miscellaneous": 0,
  "Gift Baskets and Food Gifts": 0,
  "Dietary Specific Items": 0,
  "Cooking Tools and Kitchen Goods": 0,
  "Sweeteners": 0,
  "Nuts and Seeds": 0,
  "Coffee/Tea": 0,
  "Vegetables and Beans": 0,
  "Health-Conscious Options": 0,
  "Prepared and Ready-Made Meals": 0,
  "Ethnic and Specialty Foods": 0,
  "Culinary Specialties": 0
}"""


# basically this one was extracted badly, I could probably fix the extractor function but this is simple.

{Butter & Sea Salt} 3pk" is a gourmet microwave popcorn product that is specifically designed for snacking. Given its description, it is clear that this item is intended to be consumed as a snack rather than as a meal or a cooking ingredient. 

The main category that fits this item is "Snacks" because popcorn is widely recognized as a snack food. It does not fit into categories such as "Beverages," "Cooking Ingredients," "Breakfast Foods," "Sweets and Desserts," "Health Foods," "Canned and Packaged Foods," "Baby Food," "Condiments and Sauces," "Fruits and Vegetables," "Specialty Foods," "Dried and Preserved Foods," "Grains and Pasta," "Miscellaneous," "Gift Baskets and Food Gifts," "Dietary Specific Items," "Cooking Tools and Kitchen Goods," "Sweeteners," "Nuts and Seeds," "Coffee/Tea," "Vegetables and Beans," "Health-Conscious Options," "Prepared and Ready-Made Meals," "Ethnic and Specialty Foods," or "Culinary Specialties." 

Therefore, the only applicable category for this item is "

In [28]:


electronic_category_keys = [
    'Camera and Accessories', 'Computers and Accessories', 'Audio Equipment',
    'Tablets and Accessories', 'Storage Solutions', 'Networking Equipment',
    'Mobile Devices and Accessories', 'Travel Accessories', 'Gaming',
    'Home Entertainment Systems', 'Miscellaneous Electronics', 'Power Solutions',
    'Cables and Connectors', 'Security Systems', 'Car Technology and Accessories',
    'Photography and Camera Equipment', 'Adapters and Cables',
    'Television and Accessories', 'AV Setup', 'GPS and Navigation Accessories',
    'Mobile Device Protection', 'Streaming and Media', 'PC Building and Assembly',
    'General Electronics', 'Walkie Talkies and Communication Devices'
]

electronic_category_indices = []

for i in electronic_json_categories:
    positive_indices = []
    for key, value in json.loads(i).items():
        if value == 1:
            try:
                # Find the index of this key in our master list
                index = electronic_category_keys.index(key)
                positive_indices.append(index)
            except ValueError:
                # This handles cases where the LLM hallucinates a category name
                # that doesn't exist in our master list.
                print(f"Warning: LLM returned an unknown category '{key}'")

    electronic_category_indices.append(positive_indices)


clothing_category_keys = [
    'Footwear', 'Accessories', 'Costumes and Themed Apparel',
    'Lingerie and Underwear', 'Baby and Kids Clothing', 'Activewear and Sportswear',
    'Fashion Accessories', 'Seasonal and Thematic Products', 'Electronics',
    'Children\'s Items', 'Carrying Items', 'Maintenance', 'Clothing'
]

clothing_category_indices = []

for i in clothing_json_categories:
    positive_indices = []
    for key, value in json.loads(i).items():
        if value == 1:
            try:
                # Find the index of this key in our master list
                index = clothing_category_keys.index(key)
                positive_indices.append(index)
            except ValueError:
                # This handles cases where the LLM hallucinates a category name
                # that doesn't exist in our master list.
                print(f"Warning: LLM returned an unknown category '{key}'")

    clothing_category_indices.append(positive_indices)

clothing_category_indices[4052] = [5]



food_category_keys = [
    'Snacks', 'Beverages', 'Cooking Ingredients',
    'Breakfast Foods', 'Sweets and Desserts', 'Health Foods',
    'Canned and Packaged Foods', 'Baby Food', 'Condiments and Sauces',
    'Fruits and Vegetables', 'Specialty Foods', 'Dried and Preserved Foods',
    'Grains and Pasta', 'Miscellaneous', 'Gift Baskets and Food Gifts',
    'Dietary Specific Items', 'Cooking Tools and Kitchen Goods',
    'Sweeteners', 'Nuts and Seeds', 'Coffee/Tea',
    'Vegetables and Beans', 'Health-Conscious Options', 'Prepared and Ready-Made Meals',
    'Ethnic and Specialty Foods', 'Culinary Specialties'
]

food_category_indices = []

for i in range(len(food_json_categories)):
    positive_indices = []
    # print(i)
    for key, value in json.loads(food_json_categories[i]).items():
        if value == 1:
            try:
                # Find the index of this key in our master list
                index = food_category_keys.index(key)
                positive_indices.append(index)
            except ValueError:
                # This handles cases where the LLM hallucinates a category name
                # that doesn't exist in our master list.
                print(f"Warning: LLM returned an unknown category '{key}'")

    food_category_indices.append(positive_indices)




In [ ]:
electronic_item_category_list = [[electronic_category_keys[i] for i in j] for j in electronic_category_indices]

clothing_item_category_list = [[clothing_category_keys[i] for i in j] for j in clothing_category_indices]

food_item_category_list = [[food_category_keys[i] for i in j] for j in food_category_indices]

In [ ]:
def finding_food_metadata_prompts(items, description_list, categories):

    prompt_list = []

    for i in range(len(items)):
        user_prompt = f"""
**CONTEXT:**
You have received the following item from the food domain. Your goal is to enrich the metadata of the target item.

---
**YOUR TASK:**

Item: {items[i]}
Description: {description_list[i]}
Categories: {categories[i]}

---

After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions. Do not include any other text after the separator.

**JSON Schema:**
```json
{{
  // Describes the specific type of product within a broader category.
  // Examples: "gourmet cheese", "protein bar", "spicy salsa", "frozen lasagna"
  "product_type": "string",

  // The manufacturer or brand name of the item. Put none if unsure.
  "brand": "string",

  // A list of 1-5 defining dietary features or a general description.
  // Examples: "Organic", "Gluten-Free", "Vegan", "Low-Sodium", "Non-GMO"
  "dietary_considerations": [
    "string"
  ],

  // The dominant taste characteristics of the product.
  // Examples: "savory", "sweet", "spicy", "umami", "tart"
  "flavor_profile": "string",

  // The product's market position based on price and quality.
  // Examples: "High-Cost", "Medium-Cost", "Low-Cost"
  "cost_tier": "string",

  // A list of 1-5 defining technical features or how the product is prepared.
  // Examples: "ready-to-eat", "requires cooking", "shelf-stable", "refrigerated", "single-serving"
  "key_features": [
    "string"
  ]
}}
"""

        prompt_list.append(user_prompt)

    return prompt_list


In [ ]:
def finding_clothing_metadata_prompts(items, description_list, categories):

    prompt_list = []

    for i in range(len(items)):
        user_prompt = f"""
**CONTEXT:**
You have received the following item from the clothing domain. Your goal is to enrich the metadata of the target item.

---
**YOUR TASK:**

Item: {items[i]}
Description: {description_list[i]}
Categories: {categories[i]}

---

After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions. Do not include any other text after the separator.

**JSON Schema:**
```json
{{
  // Describes the specific type of product within a broader category.
  // Examples: "denim jacket", "maxi dress", "running shorts", "crewneck sweater"
  "product_type": "string",

  // The manufacturer or brand name of the item. Put none if unsure.
  "brand": "string",

  // The primary design philosophy or key trade-off.
  // Examples: "Athleisure", "Sustainable", "Functional", "Fashion-forward", "Comfort"
  "design_focus": "string",

  // The intended audience or context for the product.
  // Examples: "Men's", "Women's", "Unisex", "Kids", "Formal", "Workwear", "Streetwear", "Casual"
  "target_user": "string",

  // The product's market position based on price and quality.
  // Examples: "High-Cost", "Medium-Cost", "Low-Cost"
  "cost_tier": "string",

  // A list of 1-5 defining technical features or specifications.
  // Examples: "waterproof", "stretch fabric", "insulated", "UPF 50+", "moisture-wicking"
  "key_features": [
    "string"
  ]
}}
"""

        prompt_list.append(user_prompt)

    return prompt_list

In [ ]:
def finding_electronic_metadata_prompts(items, description_list, categories):

    prompt_list = []

    for i in range(len(items)):
        user_prompt = f"""
**CONTEXT:**
You have received the following item from the electronics domain. Your goal is to enrich the metadata of the target item.

---
**YOUR TASK:**

Item: {items[i]}
Description: {description_list[i]}
Categories: {categories[i]}

---

After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions. Do not include any other text after the separator.

**JSON Schema:**
```json
{{
  // Describes the specific type of product within a broader category.
  "product_type": "string",

  // The manufacturer or brand name of the item. Put none if unsure.
  "brand": "string",

  // The primary design philosophy or key trade-off.
  // Examples: "Performance", "Portability", "Durability", "Value/Budget", "Aesthetics"
  "design_focus": "string",

  // The intended audience for the product.
  // Examples: "Professional", "Consumer", "Student"
  "target_user": "string",

  // The product's market position based on price and quality.
  // Examples: "High-Cost", "Medium-Cost", "Low-Cost"
  "cost_tier": "string",

  // A list of 1-3 defining technical features or specifications.
  "key_features": [
    "string"
  ]
}}
"""

        prompt_list.append(user_prompt)

    return prompt_list

In [ ]:
finding_item_electronic_metadata_prompts = finding_electronic_metadata_prompts(electronic_items, text_electronics, electronic_item_category_list)

finding_item_category_prompts = [{"prompts": data} for data in finding_item_electronic_metadata_prompts]



finding_item_clothing_metadata_prompts = finding_clothing_metadata_prompts(clothing_items, text_clothing, clothing_item_category_list)

finding_item_clothing_category_prompts = [{"prompts": data} for data in finding_item_clothing_metadata_prompts]


finding_item_food_metadata_prompts = finding_food_metadata_prompts(food_items, text_food, food_item_category_list)

finding_item_food_category_prompts = [{"prompts": data} for data in finding_item_food_metadata_prompts]


# metadata_for_each_electronic_item = await openai_request(finding_item_category_prompts, system=None, batch_size=128, delay=0)

# with open(f"/content/drive/MyDrive/BundleRec Data/item_metadata_electronic.pkl", 'wb') as f:
#     pickle.dump(metadata_for_each_electronic_item, f)

# metadata_for_each_clothing_item = await openai_request(finding_item_clothing_category_prompts, system=None, batch_size=128, delay=0)

# with open(f"/content/drive/MyDrive/BundleRec Data/item_metadata_clothing.pkl", 'wb') as f:
#     pickle.dump(metadata_for_each_clothing_item, f)

# metadata_for_each_food_item = await openai_request(finding_item_food_category_prompts, system=None, batch_size=128, delay=0)

# with open(f"/content/drive/MyDrive/BundleRec Data/item_metadata_food.pkl", 'wb') as f:
#     pickle.dump(metadata_for_each_food_item, f)

In [24]:
with open(f"/content/LLM4BEAR/BundleRec Data/item_metadata_electronic.pkl", 'rb') as f:
    electronic_item_metadata = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/item_metadata_clothing.pkl", 'rb') as f:
    clothing_item_metadata = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/item_metadata_food.pkl", 'rb') as f:
    food_item_metadata = pickle.load(f)

# Each item is decomposed into constituent characteristics, and these characteristics tokenised.

In [25]:
print(food_item_metadata[0])

To enrich the metadata for Plocky's Hummus Chips, Original, we can analyze the provided information and infer additional details based on common characteristics of similar products in the market.

1. **Product Type**: The item is a type of snack chip made from hummus, which positions it within the broader category of healthier snack alternatives. Therefore, the product type can be defined as "hummus chips".

2. **Brand**: The brand name is clearly stated as "Plocky's".

3. **Dietary Considerations**: Given that these chips are made from hummus, they are likely to be gluten-free and may also be vegan, as hummus is typically made from chickpeas and does not contain animal products. Therefore, the dietary considerations can include "Gluten-Free" and "Vegan".

4. **Flavor Profile**: While the description does not specify the flavor, hummus chips generally have a savory flavor profile, which is characteristic of chickpeas and the spices often used in hummus.

5. **Cost Tier**: The cost tier

In [26]:
metadata_electronic_jsons = [extract_json_from_response(i) for i in electronic_item_metadata]

metadata_clothing_jsons = [extract_json_from_response(i) for i in clothing_item_metadata]

metadata_food_jsons = [extract_json_from_response(i) for i in food_item_metadata]


metadata_clothing_jsons[3294] = """{
  "product_type": "pajama set",
  "brand": "Leveret",
  "design_focus": "Comfort",
  "target_user": "Kids",
  "cost_tier": "Low-Cost",
  "key_features": [
    "100% cotton"
  ]
}"""

metadata_clothing_jsons[4393] = """{
  "product_type": "baby bodysuit and shoe set",
  "brand": "none",
  "design_focus": "Fashion-forward",
  "target_user": "Kids",
  "cost_tier": "Medium-Cost",
  "key_features": [
    "soft fabric",
    "easy to put on"
  ]
}"""


metadata_clothing_jsons[4484] = """{
  "product_type": "ankle boot",
  "brand": "none",
  "design_focus": "Fashion-forward",
  "target_user": "Women's",
  "cost_tier": "Medium-Cost",
  "key_features": [
    "Western-inspired design",
    "Ankle height",
    "Stylish",
    "Versatile for casual or dressy occasions",
    "Comfortable fit"
  ]
}"""


metadata_food_jsons[363] = """{
  "product_type": "key lime pie mix",
  "brand": "none",
  "dietary_considerations": [],
  "flavor_profile": "sweet and tangy",
  "cost_tier": "Medium-Cost",
  "key_features": [
    "requires preparation",
    "shelf-stable"
  ]
}"""


metadata_food_jsons[492] = """{
  "product_type": "spice blend",
  "brand": "none",
  "dietary_considerations": [
    "none"
  ],
  "flavor_profile": "spicy",
  "cost_tier": "Medium-Cost",
  "key_features": [
    "shelf-stable",
    "requires cooking"
  ]
}"""


metadata_food_jsons[498] = """{
  "product_type": "fruit gummy snacks",
  "brand": "Tasty Brand",
  "dietary_considerations": [
    "Organic"
  ],
  "flavor_profile": "sweet",
  "cost_tier": "Medium-Cost",
  "key_features": [
    "individually packaged",
    "made with real fruit"
  ]
}"""


metadata_food_jsons[1329] = """{
  "product_type": "brewed chocolate beverage",
  "brand": "Choffy",
  "dietary_considerations": [
    "Caffeine-Free"
  ],
  "flavor_profile": "rich, coffee-like",
  "cost_tier": "Medium-Cost",
  "key_features": [
    "brewed",
    "shelf-stable"
  ]
}"""


metadata_food_jsons[1532] = """{
  "product_type": "coffee pods",
  "brand": "Café Escapes",
  "dietary_considerations": ["none"],
  "flavor_profile": "sweet",
  "cost_tier": "Medium-Cost",
  "key_features": ["single-serving", "ready-to-brew"]
}"""


metadata_food_jsons[1653] = """{
  "product_type": "meat substitute",
  "brand": "none",
  "dietary_considerations": [
    "Vegan"
  ],
  "flavor_profile": "savory",
  "cost_tier": "Medium-Cost",
  "key_features": [
    "requires cooking"
  ]
}"""


metadata_food_jsons[1784] = """{
  "product_type": "snack mix",
  "brand": "none",
  "dietary_considerations": [],
  "flavor_profile": "savory",
  "cost_tier": "Medium-Cost",
  "key_features": [
    "ready-to-eat",
    "shelf-stable"
  ]
}"""


metadata_food_jsons[2487] = """{
  "product_type": "assorted potato snacks",
  "brand": "none",
  "dietary_considerations": [
    "none"
  ],
  "flavor_profile": "savory",
  "cost_tier": "Medium-Cost",
  "key_features": [
    "single-serving",
    "ready-to-eat",
    "shelf-stable"
  ]
}"""

metadata_food_jsons[3145] = """{
  "product_type": "liquid sweetener",
  "brand": "none",
  "dietary_considerations": [
    "None"
  ],
  "flavor_profile": "sweet",
  "cost_tier": "Low-Cost",
  "key_features": [
    "shelf-stable",
    "ready-to-use"
  ]
}"""


metadata_food_jsons[3171] = """{
  "product_type": "chewing gum",
  "brand": "none",
  "dietary_considerations": [],
  "flavor_profile": "sweet",
  "cost_tier": "Low-Cost",
  "key_features": [
    "ready-to-eat",
    "shelf-stable"
  ]
}"""


metadata_food_jsons[3255] = """{
  "product_type": "gourmet microwave popcorn",
  "brand": "Quinn",
  "dietary_considerations": ["none"],
  "flavor_profile": "savory",
  "cost_tier": "Medium-Cost",
  "key_features": [
    "ready-to-eat",
    "pack of three bags"
  ]
}"""

metadata_food_jsons[3750] = """{
  "product_type": "gourmet snack box",
  "brand": "none",
  "dietary_considerations": [],
  "flavor_profile": "varied",
  "cost_tier": "High-Cost",
  "key_features": [
    "ready-to-eat",
    "shelf-stable"
  ]
}"""

In [ ]:
# with open(f"/content/drive/MyDrive/BundleRec Data/specific_electronic_product_metadata.pkl", 'wb') as f:
#     pickle.dump(metadata_electronic_jsons, f)
# with open(f"/content/drive/MyDrive/BundleRec Data/specific_clothing_product_metadata.pkl", 'wb') as f:
#     pickle.dump(metadata_clothing_jsons, f)
# with open(f"/content/drive/MyDrive/BundleRec Data/specific_food_product_metadata.pkl", 'wb') as f:
#     pickle.dump(metadata_food_jsons, f)

In [29]:
food_sub_categories = [[] for i in range(25)]

for i in range(len(metadata_food_jsons)):
    # print(i)
    metadata_json = json.loads(metadata_food_jsons[i])

    for j in food_category_indices[i]:
        try:
            food_sub_categories[j].append(metadata_json['product_type'])
        except:
            print("no product ??")

In [30]:
electronic_sub_categories = [[] for i in range(25)]

for i in range(len(metadata_electronic_jsons)):
    metadata_json = json.loads(metadata_electronic_jsons[i])

    for j in electronic_category_indices[i]:
        try:
            electronic_sub_categories[j].append(metadata_json['product_type'])
        except:
            print("no product ??")

In [31]:
clothing_sub_categories = [[] for i in range(13)]

for i in range(len(metadata_clothing_jsons)):
    # print(i)
    metadata_json = json.loads(metadata_clothing_jsons[i])

    for j in clothing_category_indices[i]:
        try:
            clothing_sub_categories[j].append(metadata_json['product_type'])
        except:
            print("no product ??")

In [32]:
food_sub_categories = [list(set(sublist)) for sublist in food_sub_categories]

electronic_sub_categories = [list(set(sublist)) for sublist in electronic_sub_categories]

clothing_sub_categories = [list(set(sublist)) for sublist in clothing_sub_categories]

In [34]:
def making_grouped_products_prompts(sub_categories, category_keys):

    prompt_list = []
    for i in range(len(sub_categories)):
        product_types_str = '\n'.join([f"- {j}" for j in sub_categories[i]])
        user_prompt = f"""
**CONTEXT:**
You are a data analyst specializing in product taxonomy. You have received the following list of raw product types from the '{category_keys[i]}' category. The list is redundant and needs to be cleaned and summarized.

**Raw Product Types:**
{product_types_str}

---
**YOUR TASK:**
Your goal is to analyze all the raw product types and condense them into a generalized list of core groups.
1. Group similar and synonymous items under a single, representative name (e.g., group 'Wireless Headphones' and 'Bluetooth Headset' under 'Wireless Headphones').
2. Return the final list of these groups, ordered by the frequency of their appearance in the raw list above. The most common groups should be at the top.

After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions. Do not include any other text after the separator.

**JSON Schema:**
```json
{{
  "product_types": [
    "string // The most frequent product group",
    "string // The second most frequent product group",
    "..."
  ]
}}
"""

        prompt_list.append(user_prompt)

    return prompt_list

# There are many product types with the same meaning, so we'll standardise them, and then have the LLM choose from a list of potential product types.

In [35]:
finding_product_prompts = making_grouped_products_prompts(electronic_sub_categories, electronic_category_keys)

print(finding_product_prompts[0])

finding_product_prompts = [{"prompts": data} for data in finding_product_prompts]


finding_product_prompts = making_grouped_products_prompts(clothing_sub_categories, clothing_category_keys)

# print(finding_product_prompts[0])

finding_product_prompts = [{"prompts": data} for data in finding_product_prompts]

finding_product_prompts = making_grouped_products_prompts(food_sub_categories, food_category_keys)

# print(finding_product_prompts[0])

finding_product_prompts = [{"prompts": data} for data in finding_product_prompts]



# product_types_for_electronic_domain = await openai_request(finding_product_prompts, system=None, batch_size=128, delay=0)
# product_types_for_clothing_domain = await openai_request(finding_product_prompts, system=None, batch_size=128, delay=0)
# product_types_for_food_domain = await openai_request(finding_product_prompts, system=None, batch_size=128, delay=0)




**CONTEXT:**
You are a data analyst specializing in product taxonomy. You have received the following list of raw product types from the 'Camera and Accessories' category. The list is redundant and needs to be cleaned and summarized.

**Raw Product Types:**
- Flash Diffuser
- UV filter
- Extension Tube Set
- Air Blower
- Compact Flash Card Storage Wallet
- Accessory
- Flash Bounce Diffuser
- Light Stand
- Underwater Slave Flash
- Stereo Microphone
- Power Adapter and Coupler Kit
- Reflector
- External Viewfinder
- Flash Hot Shoe Mount Adapter
- Audio/Video Cable
- soft white diffuser umbrella
- Rechargeable Battery
- CCTV Cable Extension Kit
- Circular Polarizer Filter
- Filter System Kit
- Light Stand Accessory
- Camera Holster System
- Wireless Remote Shutter Release Cable
- Studio Flash Light
- Lens
- Tablet
- Color Correction Tool
- Focal Reducer Speed Booster Lens Turbo Adapter
- Video Head
- Eyecup
- Security Monitoring System
- Wireless Shutter Remote Control
- Speedlite
- Came

In [ ]:
"""===JSON_START===
```json
{
  "product_types": [
    "Snacks",
    "Sauces and Condiments",
    "Spices and Seasonings",
    "Beverages",
    "Baked Goods and Desserts",
    "Prepared Foods",
    "Fruits and Nuts",
    "Miscellaneous"
  ]
}"""

# Product Types will look like this.

# Logic for Manually adding these product types can be found below, where each Food Category corresponds to a list of product types.

In [43]:
food_product_types = [extract_json_from_response(i) for i in product_types_for_food_domain]

electronic_product_types = [extract_json_from_response(i) for i in product_types_for_electronic_domain]

clothing_product_types = [extract_json_from_response(i) for i in product_types_for_clothing_domain]

# I guess i didn't save this in the past, so you'll have to trust that it's legit.


food_all_product_types = [[] for i in range(25)]

for i in range(len(food_product_types)):
    # print(i)
    product_types_json = json.loads(food_product_types[i])
    food_all_product_types[i] = product_types_json['product_types']

food_all_product_types[1].append("Nutritional Shake")
food_all_product_types[5].append("Nutritional Shake")
food_all_product_types[1].append("Hydration Mix")
food_all_product_types[5].append("Hydration Mix")
food_all_product_types[0].append("Baby Snacks")
food_all_product_types[12].append("Crispbread")
food_all_product_types[11].append("Dried Tomatoes")

# These products are supposed to appear under a certain Food category, i.e., Nutritional Shake was not placed in food category 1.


# with open(f"/content/drive/MyDrive/BundleRec Data/all_product_types_food.pkl", 'wb') as f:
#     pickle.dump(food_all_product_types, f)


clothing_all_product_types = [[] for i in range(13)]

for i in range(len(clothing_product_types)):
    product_types_json = json.loads(clothing_product_types[i])
    clothing_all_product_types[i] = product_types_json['product_types']


clothing_all_product_types[4].append("Footie")
clothing_all_product_types[1].append("Wristlets")
clothing_all_product_types[10].append("Wristlet")
clothing_all_product_types[3].append("Bodysuit")
clothing_all_product_types[1].append("Cufflinks")
clothing_all_product_types[6].append("Cufflinks")


# with open(f"/content/drive/MyDrive/BundleRec Data/all_product_types_clothing.pkl", 'wb') as f:
#     pickle.dump(clothing_all_product_types, f)


electronic_all_product_types = [[] for i in range(25)]

for i in range(len(electronic_product_types)):
    product_types_json = json.loads(electronic_product_types[i])
    electronic_all_product_types[i] = product_types_json['product_types']

electronic_all_product_types[1].append("Motherboard")
electronic_all_product_types[8].append("Motherboard")
electronic_all_product_types[22].append("Motherboard")
electronic_all_product_types[14].append("Amplifier Installation Kit")
electronic_all_product_types[2].append("Headset")
electronic_all_product_types[9].append("Projector Screens")
electronic_all_product_types[1].append("Sound Card")
electronic_all_product_types[1].append("Bluetooth Adapter")



# with open(f"/content/drive/MyDrive/BundleRec Data/all_product_types_electronic.pkl", 'wb') as f:
#     pickle.dump(electronic_all_product_types, f)

NameError: name 'product_types_for_food_domain' is not defined

In [38]:
with open(f"/content/LLM4BEAR/BundleRec Data/all_product_types_food.pkl", 'rb') as f:
    food_guys = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/all_product_types_electronic.pkl", 'rb') as f:
    electronic_guys = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/all_product_types_clothing.pkl", 'rb') as f:
    clothing_guys = pickle.load(f)

In [41]:
for i in range(len(food_category_keys)):
    print(food_category_keys[i])
    print("="*20)
    for j in food_guys[i]:
        print(j)

    print()
    print("="*50)
    print()

Snacks
Chocolate
Snack Bars
Nuts
Cookies
Chips
Gummy Candy
Popcorn
Jerky
Fruit Snacks
Crackers
Seasoned Snacks
Hard Candy
Cereal Snacks
Dried Fruit
Baked Goods
Cheese Snacks
Sugar-Free Snacks
Specialty Snacks
Beverages
Miscellaneous Snacks
Baby Snacks


Beverages
Coffee Products
Tea Products
Juice Products
Smoothies and Mixes
Soda and Carbonated Beverages
Specialty Beverages
Milk and Dairy Alternatives
Miscellaneous
Nutritional Shake
Hydration Mix


Cooking Ingredients
Seasoning/Spice
Flour
Sugar/Sweeteners
Baking Mixes
Oils/Fats
Nuts/Seeds
Canned Goods
Dried Vegetables/Legumes
Sauces/Pastes
Powders
Mixes
Extracts/Essences
Dairy Products
Grains/Rice
Chocolate Products
Beverages
Miscellaneous


Breakfast Foods
Oatmeal and Hot Cereals
Breakfast Cereals
Pancake and Waffle Mixes
Granola and Snack Bars
Coffee and Beverages
Spreads and Syrups
Baking Mixes and Bread
Muffin and Cookie Mixes
Frozen and Canned Foods
Miscellaneous Breakfast Items


Sweets and Desserts
Chocolate
Cookies
Candy
Syru

In [42]:
food_relevant_products = []

for i in range(len(text_food)):
    product_type_strings = ""
    for j in food_category_indices[i]:
        product_type_list = food_all_product_types[j]
        for k in product_type_list:
            product_type_strings += f"- {k}\n"
    food_relevant_products.append(product_type_strings)


clothing_relevant_products = []

for i in range(len(text_clothing)):
    product_type_strings = ""
    for j in clothing_category_indices[i]:
        product_type_list = clothing_all_product_types[j]
        for k in product_type_list:
            product_type_strings += f"- {k}\n"
    clothing_relevant_products.append(product_type_strings)

electronic_relevant_products = []

for i in range(len(text_electronics)):
    product_type_strings = ""
    for j in electronic_category_indices[i]:
        product_type_list = electronic_all_product_types[j]
        for k in product_type_list:
            product_type_strings += f"- {k}\n"
    electronic_relevant_products.append(product_type_strings)

NameError: name 'all_product_types' is not defined

In [ ]:
# Note: The function now takes a single, master list of product types
def relevant_food_products_prompts(items, description_list, metadata, product_type_master_list):

    prompt_list = []
    # Convert the master list into a formatted string once, before the loop.

    for i in range(len(items)):
        product_types_str = product_type_master_list[i]
        user_prompt = f"""
**CONTEXT:**
You are an expert product classifier. Your goal is to assign the given food item to the single most relevant product type from the master list provided.

**Item to Classify:**
- Item Name: {items[i]}
- Description: {description_list[i]}
- Metadata:
{metadata[i]}

---
**YOUR TASK:**
Analyze the item and choose the **ONE and ONLY ONE** product type from the list below that best describes it.

**List of Product Types:**
{product_types_str}

After your analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object with your conclusion.

**JSON Schema:**
```json
{{
  "product_type": "string" // You MUST choose from the product list and it must be word for word.
}}
"""
        prompt_list.append(user_prompt)

    return prompt_list

# Note: The function now takes a single, master list of product types
def relevant_clothing_products_prompts(items, description_list, metadata, product_type_master_list):

    prompt_list = []
    # Convert the master list into a formatted string once, before the loop.

    for i in range(len(items)):
        product_types_str = product_type_master_list[i]
        user_prompt = f"""
**CONTEXT:**
You are an expert product classifier. Your goal is to assign the given clothing item to the single most relevant product type from the master list provided.

**Item to Classify:**
- Item Name: {items[i]}
- Description: {description_list[i]}
- Metadata:
{metadata[i]}

---
**YOUR TASK:**
Analyze the item and choose the **ONE and ONLY ONE** product type from the list below that best describes it.

**List of Product Types:**
{product_types_str}

After your analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object with your conclusion.

**JSON Schema:**
```json
{{
  "product_type": "string" // You MUST choose from the product list and it must be word for word.
}}
"""
        prompt_list.append(user_prompt)

    return prompt_list

# Note: The function now takes a single, master list of product types
def relevant_electronic_products_prompts(items, description_list, metadata, product_type_master_list):

    prompt_list = []
    # Convert the master list into a formatted string once, before the loop.

    for i in range(len(items)):
        product_types_str = product_type_master_list[i]
        user_prompt = f"""
**CONTEXT:**
You are an expert product classifier. Your goal is to assign the given electronic item to the single most relevant product type from the master list provided.

**Item to Classify:**
- Item Name: {items[i]}
- Description: {description_list[i]}
- Metadata:
{metadata[i]}

---
**YOUR TASK:**
Analyze the item and choose the **ONE and ONLY ONE** product type from the list below that best describes it.

**List of Product Types:**
{product_types_str}

After your analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object with your conclusion.

**JSON Schema:**
```json
{{
  "product_type": "string" // You MUST choose from the product list and it must be word for word.
}}
"""
        prompt_list.append(user_prompt)

    return prompt_list


In [ ]:
food_relevant_product_prompts = relevant_food_products_prompts(food_items, text_food, food_item_metadata, food_relevant_products)

print(food_relevant_product_prompts[0])

food_relevant_product_prompts = [{"prompts": data} for data in food_relevant_product_prompts]

clothing_relevant_product_prompts = relevant_clothing_products_prompts(clothing_items, text_clothing, clothing_item_metadata, clothing_relevant_products)

print(clothing_relevant_product_prompts[0])

clothing_relevant_product_prompts = [{"prompts": data} for data in clothing_relevant_product_prompts]


electronic_relevant_product_prompts = relevant_electronic_products_prompts(clothing_items, text_clothing, clothing_item_metadata, electronic_relevant_products)

print(electronic_relevant_product_prompts[0])

relevant_product_prompts = [{"prompts": data} for data in electronic_relevant_product_prompts]

# single_product_type_for_each_item = await openai_request(food_relevant_product_prompts, system=None, batch_size=128, delay=0)

# with open(f"/content/drive/MyDrive/BundleRec Data/food_product_type.pkl", 'wb') as f:
#     pickle.dump(single_product_type_for_each_item, f)


# single_product_type_for_each_item = await openai_request(clothing_relevant_product_prompts, system=None, batch_size=128, delay=0)

# with open(f"/content/drive/MyDrive/BundleRec Data/clothing_product_type.pkl", 'wb') as f:
#     pickle.dump(single_product_type_for_each_item, f)


# single_product_type_for_each_item = await openai_request(electronic_relevant_product_prompts, system=None, batch_size=128, delay=0)

# with open(f"/content/drive/MyDrive/BundleRec Data/electronic_product_type.pkl", 'wb') as f:
#     pickle.dump(single_product_type_for_each_item, f)

In [45]:
with open(f"/content/LLM4BEAR/BundleRec Data/electronic_product_type.pkl", 'rb') as f:
    electronic_products = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/clothing_product_type.pkl", 'rb') as f:
    clothing_products = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/food_product_type.pkl", 'rb') as f:
    food_products = pickle.load(f)

In [47]:
print(len(electronic_products))
print(electronic_products[0])

3499
===JSON_START===
{
  "product_type": "Graphics Card"
}


In [60]:
food_product_set = []

for i in range(len(text_food)):
    product_types = []
    for j in food_category_indices[i]:
        product_types = product_types + food_guys[j]
    food_product_set.append(product_types)

food_problems = []


for i in range(len(food_products)):
    specific_product = json.loads(extract_json_from_response(food_products[i]))['product_type']
    categories = food_category_indices[i]
    category_str = [food_category_keys[j] for j in categories]
    if specific_product not in food_product_set[i]:
        print(i, specific_product, categories, category_str)
        print(text_food[i])
        print()

        food_problems.append(i)

564 sauce mix [2, 8] ['Cooking Ingredients', 'Condiments and Sauces']
A blend of spices and seasonings designed to create flavorful enchilada sauce, offered in a convenient three-pack for easy meal preparation.

573 Toddler Snacks [0, 7, 9] ['Snacks', 'Baby Food', 'Fruits and Vegetables']
A pack of seven 1-ounce bags containing melt-in-your-mouth fruit and vegetable snacks designed for toddlers, featuring a tropical flavor blend.

1309 Tortilla [5, 12, 15, 21] ['Health Foods', 'Grains and Pasta', 'Dietary Specific Items', 'Health-Conscious Options']
These tortillas are a low-carb option made with flax, oat bran, and whole wheat, suitable for wraps or as a side in various meals.

1350 Snack Mix [0, 4] ['Snacks', 'Sweets and Desserts']
A 30-ounce bag of a crunchy snack mix featuring peanut-flavored chocolate candies combined with assorted nuts and other sweet treats.

2317 brown sugar cubes [4, 17] ['Sweets and Desserts', 'Sweeteners']
These brown sugar cubes are perfect for sweetening b

In [61]:
electronic_product_set = []

for i in range(len(text_electronics)):
    product_types = []
    for j in electronic_category_indices[i]:
        product_types = product_types + electronic_guys[j]
    electronic_product_set.append(product_types)

electronic_problems = []


for i in range(len(electronic_products)):
    specific_product = json.loads(extract_json_from_response(electronic_products[i]))['product_type']
    categories = electronic_category_indices[i]
    category_str = [electronic_category_keys[j] for j in categories]
    if specific_product not in electronic_product_set[i]:
        print(i, specific_product, categories, category_str)
        print(text_electronics[i])
        print()

        electronic_problems.append(i)

171 DVR Security Camera System [0, 13] ['Camera and Accessories', 'Security Systems']
An 8-channel DVR security system with four high-resolution outdoor cameras for day and night surveillance, excluding a hard drive.

489 Expansion Card [1, 12] ['Computers and Accessories', 'Cables and Connectors']
This is a computer expansion card that adds four RS232 serial ports to a system via a PCI-Express x1 slot, utilizing the Moschip 9904 chipset and includes a fan-out cable for connectivity.

504 Microphone Preamp [2, 10] ['Audio Equipment', 'Miscellaneous Electronics']
A compact device that enhances microphone audio quality by amplifying signals, providing professional-grade sound for recording and live performances.

722 Solid-State Drive [1, 4, 22] ['Computers and Accessories', 'Storage Solutions', 'PC Building and Assembly']
This is an 80 GB internal solid-state drive designed for improving computer storage performance with faster data access and reliability, using a SATA 3.0 Gb/s interfac

In [62]:
clothing_product_set = []

for i in range(len(text_clothing)):
    product_types = []
    for j in clothing_category_indices[i]:
        product_types = product_types + clothing_guys[j]
    clothing_product_set.append(product_types)

clothing_problems = []


for i in range(len(clothing_products)):
    specific_product = json.loads(extract_json_from_response(clothing_products[i]))['product_type']
    categories = clothing_category_indices[i]
    category_str = [clothing_category_keys[j] for j in categories]
    if specific_product not in clothing_product_set[i]:
        print(i, specific_product, categories, category_str)
        print(text_clothing[i])
        print()

        clothing_problems.append(i)

78 Crossbody Bag [1, 6, 10] ['Accessories', 'Fashion Accessories', 'Carrying Items']
This crossbody bag features a zippered closure and adjustable strap, designed for stylish and convenient hands-free carrying in a vibrant Miami Blue color.

150 shorts [5, 12] ['Activewear and Sportswear', 'Clothing']
These cargo shorts feature a durable ripstop fabric, multiple pockets for utility, and a bright lemon color, designed for comfort and outdoor activities.

321 clog [0] ['Footwear']
This is a pair of women's leather clogs designed for comfort and ease of wear, featuring a classic slip-on style suitable for casual outings.

500 swimsuit [5] ['Activewear and Sportswear']
This swim tee is designed for men in extended sizes, featuring UPF 50+ protection to shield against harmful UV rays while swimming or engaging in water activities.

1157 jewelry set [1, 6] ['Accessories', 'Fashion Accessories']
This jewelry set features a silver owl pendant on a 28-inch adjustable chain, accompanied by match

In [63]:
food_problem_titles = ["Sauce", "Baby Snacks", "Chips and Crisps", "Nuts", "Sugars", "Sweeteners"]

for i in range(len(food_problems)):
    print(food_problems[i], food_problem_titles[i])

564 Sauce
573 Baby Snacks
1309 Chips and Crisps
1350 Nuts
2317 Sugars
3145 Sweeteners


In [59]:
clothing_problem_titles = ["Bags", "Shorts", "Clog", "Swimsuit", "Body Jewelry",
                  "Shoe", "Bags", "Sandal", "Body Jewelry", "Shorts",
                  "Swimsuit", "Swimsuit", "Boot", "Shorts", "Slip-On Shoe",
                  "Slipper", "Bags", "Romper/Jumpsuit", "Bags", "Bodysuit",
                  "Slip-On Shoe", "Body Jewelry", "Shoe", "Swimsuit"]

for i in range(len(clothing_problems)):
    print(clothing_problems[i], clothing_problem_titles[i])

78 Bags
150 Shorts
321 Clog
500 Swimsuit
1157 Body Jewelry
1195 Shoe
1203 Bags
1403 Sandal
1458 Body Jewelry
1687 Shorts
1696 Swimsuit
1823 Swimsuit
1825 Boot
1901 Shorts
2182 Slip-On Shoe
2456 Slipper
2701 Bags
2964 Romper/Jumpsuit
3324 Bags
3443 Bodysuit
3651 Slip-On Shoe
3919 Body Jewelry
3987 Shoe
4052 Swimsuit


# Manually label the product types that the LLM did not label according to the given product types.

# Apparently, I just had the LLM just do it again for the Electronic domain instead of labeling it myself.

In [ ]:
problem_product_prompts = [relevant_product_prompts[i] for i in electronic_problems]

# product_type_for_problem_item = await openai_request(problem_product_prompts, system=None, batch_size=128, delay=0)



general_electronic_product_list = []

for i in range(len(text_electronics)):
    specific_product = json.loads(extract_json_from_response(electronic_products[i]))['product_type']

    if specific_product not in electronic_product_set[i]:
        general_electronic_product_list.append("")

    else:
        general_electronic_product_list.append(specific_product)

for i in range(len(product_type_for_problem_item)):
    actual_indice = electronic_problems[i]
    specific_product = json.loads(extract_json_from_response(product_type_for_problem_item[i]))['product_type']

    general_electronic_product_list[actual_indice] = specific_product

# with open(f"/content/drive/MyDrive/BundleRec Data/general_electronic_product_list.pkl", 'wb') as f:
#     pickle.dump(general_electronic_product_list, f)

In [ ]:

general_clothing_product_list = []

for i in range(len(text_clothing)):
    general_clothing_product_list.append(json.loads(extract_json_from_response(clothing_products[i]))['product_type'])

for i in range(len(clothing_problems)):
    actual_indice = clothing_problems[i]
    general_clothing_product_list[actual_indice] = clothing_problem_titles[i]


# with open(f"/content/drive/MyDrive/BundleRec Data/general_clothing_product_list.pkl", 'wb') as f:
#     pickle.dump(general_clothing_product_list, f)

In [ ]:

general_food_product_list = []

for i in range(len(text_food)):
    general_food_product_list.append(json.loads(extract_json_from_response(food_products[i]))['product_type'])

for i in range(len(food_problems)):
    actual_indice = food_problems[i]
    general_food_product_list[actual_indice] = food_problem_titles[i]

# with open(f"/content/drive/MyDrive/BundleRec Data/general_food_product_list.pkl", 'wb') as f:
#     pickle.dump(general_food_product_list, f)

# Now I have each item labeled with a specific-curated product type. Below is a non-curated product type.

In [ ]:
"""{
  "product_type": "pull-apart gummy candy",
  "brand": "Twizzlers",
  "dietary_considerations": [
    "None"
  ],
  "flavor_profile": "sweet",
  "cost_tier": "Low-Cost",
  "key_features": [
    "pull-apart",
    "shelf-stable",
    "ready-to-eat"
  ]"""